# Demo: Single-Feature and Progressive Feature Analysis
## UPDRS3 Alone vs. Top 1-5 Most Important Features

**Location**: `/Users/hc/Documents/research/Projects/PPMI/paper/demos/demo_sign.ipynb`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print('Loading data...')

In [ ]:
# Load data from absolute path
data_path = '/Users/hc/Documents/research/Projects/PPMI/PPMI_Curated_Data_Cut_Public_20250321/20250310-Table 1.csv'
df = pd.read_csv(data_path, low_memory=False)

# Filter to 3-class
df_3class = df[df['COHORT'].isin([1, 2, 3])].copy()
df_3class['diagnosis'] = df_3class['COHORT'].map({1: 'PD', 2: 'HC', 3: 'SWEDD'})

print(f'3-class dataset: {len(df_3class):,} visits')
print(f'Class distribution:')
for diag in ['PD', 'HC', 'SWEDD']:
    count = len(df_3class[df_3class['diagnosis'] == diag])
    pct = count / len(df_3class) * 100
    print(f'  {diag}: {count:,} visits ({pct:.1f}%)')

In [ ]:
# Feature sets
feature_sets = {
    'Feature 1: UPDRS3 Only': ['updrs3_score'],
    'Features 1-2': ['updrs3_score', 'updrs_totscore'],
    'Features 1-3': ['updrs3_score', 'updrs_totscore', 'Stage_partial_UPDRS1'],
    'Features 1-4': ['updrs3_score', 'updrs_totscore', 'Stage_partial_UPDRS1', 'updrs4_score'],
    'Features 1-5': ['updrs3_score', 'updrs_totscore', 'Stage_partial_UPDRS1', 'updrs4_score', 'updrs_totscore_on']
}

print('Feature progression (based on demo2 SHAP importance):')
for test_name in feature_sets.keys():
    print(f'  - {test_name}')

In [ ]:
# Setup
models = {
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print('Running tests... this may take a few minutes')

for test_name, features in feature_sets.items():
    print(f'\n{test_name}')
    
    # Complete cases only
    df_complete = df_3class[df_3class[features].notna().all(axis=1)].copy()
    
    y = LabelEncoder().fit_transform(df_complete['diagnosis'])
    X = df_complete[features].copy()
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features)
    
    print(f'  Samples: {len(df_complete):,} (PD: {len(df_complete[df_complete.diagnosis=="PD"]):,}, HC: {len(df_complete[df_complete.diagnosis=="HC"]):,}, SWEDD: {len(df_complete[df_complete.diagnosis=="SWEDD"]):,})')
    
    test_results = {}
    
    for model_name, model in models.items():
        cv_results = cross_validate(model, X_scaled, y, cv=cv, scoring='accuracy')
        acc_mean = cv_results['test_score'].mean()
        acc_std = cv_results['test_score'].std()
        
        sensitivities = []
        for train_idx, test_idx in cv.split(X_scaled, y):
            X_train, X_test = X_scaled.iloc[train_idx], X_scaled.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            
            model_clone = model.__class__(**model.get_params())
            model_clone.fit(X_train, y_train)
            y_pred = model_clone.predict(X_test)
            
            cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
            tp = cm[0, 0]
            fn = cm[0, 1] + cm[0, 2]
            sens = tp / (tp + fn) if (tp + fn) > 0 else 0
            sensitivities.append(sens)
        
        sens_mean = np.mean(sensitivities)
        sens_std = np.std(sensitivities)
        
        test_results[model_name] = {'accuracy': acc_mean, 'accuracy_std': acc_std, 'sensitivity': sens_mean, 'sensitivity_std': sens_std, 'n_samples': len(df_complete)}
        print(f'  {model_name}: Acc={acc_mean*100:.2f}%, Sens={sens_mean*100:.2f}%')
    
    results[test_name] = test_results

In [ ]:
print('\n' + '='*100)
print('RESULTS SUMMARY (Gradient Boosting)')
print('='*100)
print(f"{'Test':<30} {'Accuracy':<20} {'PD Sensitivity':<20} {'Samples':>10}")
print('-'*80)

for test_name, test_results in results.items():
    gb = test_results['Gradient Boosting']
    acc = f"{gb['accuracy']*100:.2f} +/- {gb['accuracy_std']*100:.2f}%"
    sens = f"{gb['sensitivity']*100:.2f} +/- {gb['sensitivity_std']*100:.2f}%"
    print(f'{test_name:<30} {acc:<20} {sens:<20} {gb["n_samples"]:>10,}')

print('='*100)

In [ ]:
# Key findings
test_names = list(results.keys())
test1 = results[test_names[0]]['Gradient Boosting']
test5 = results[test_names[-1]]['Gradient Boosting']

print('\n' + '='*100)
print('KEY FINDINGS')
print('='*100)

print(f'\n1. UPDRS3 ALONE:')
print(f'   Accuracy: {test1["accuracy"]*100:.2f}%')
print(f'   PD Sensitivity: {test1["sensitivity"]*100:.2f}%')
print(f'   Samples: {test1["n_samples"]:,}')

print(f'\n2. TOP 5 FEATURES:')
print(f'   Accuracy: {test5["accuracy"]*100:.2f}%')
print(f'   PD Sensitivity: {test5["sensitivity"]*100:.2f}%')
print(f'   Samples: {test5["n_samples"]:,}')

print(f'\n3. IMPROVEMENT:')
print(f'   Accuracy gain: {(test5["accuracy"] - test1["accuracy"])*100:.2f} pp')
print(f'   Sensitivity gain: {(test5["sensitivity"] - test1["sensitivity"])*100:.2f} pp')

print(f'\n4. CONCLUSION:')
if abs(test5['sensitivity'] - test1['sensitivity']) < 0.02:
    print(f'   UPDRS3 alone is SUFFICIENT - minimal improvement with more features')
else:
    print(f'   Additional features provide meaningful improvement')

print('\n' + '='*100)